# 🏙️ Multimodal Urban Event Prediction — Model Training (Graduate Level)
**Deliverable 2:** Full training pipeline with rigorous evaluation

---
### Notebook Structure
1. Environment Setup
2. Data Loading & Feature Engineering
3. Train/Test Split (Time-Series)
4. **Model Comparison** — Ridge / Random Forest / LightGBM / XGBoost
5. **TimeSeriesSplit Cross-Validation**
6. **Hyperparameter Tuning** (Optuna)
7. Final Model Evaluation & Plots
8. **Ablation Study** — contribution of each data modality
9. **Error Analysis** — when does the model fail?
10. SHAP Interpretability
11. Save Model

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import joblib
from pathlib import Path
import requests

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print("⚠️  optuna not installed — run: pip install optuna")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️  shap not installed — run: pip install shap")

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
sns.set_style('whitegrid')

# ── Paths ──────────────────────────────────────────────────────────
DATA_DIR    = Path(r"C:\Users\86188\urban_event_prediction\data")
RESULTS_DIR = Path(r"C:\Users\86188\urban_event_prediction\results")
MODELS_DIR  = Path(r"C:\Users\86188\urban_event_prediction\models")
for d in [DATA_DIR, RESULTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TAXI_PATH    = Path(r"C:\Users\86188\Downloads\yellow_tripdata_2026-01.parquet")
EVENTS_PATH  = Path(r"C:\Users\86188\Downloads\NYCHA_Citywide_Special_Events_20260413.csv")
REDDIT_PATH  = Path(r"C:\Users\86188\Downloads\Reddit_Data.csv\Reddit_Data.csv")
TWITTER_PATH = Path(r"C:\Users\86188\Downloads\Twitter_Data.csv\Twitter_Data.csv")

print("✅ Environment ready")
print(f"   XGBoost  : {xgb.__version__}")
print(f"   LightGBM : {lgb.__version__}")

## 2. Data Loading & Feature Engineering

In [ ]:
# ── 2a. Taxi → hourly demand ───────────────────────────────────────
print("Loading taxi data...")
taxi = pd.read_parquet(TAXI_PATH)
taxi["tpep_pickup_datetime"] = pd.to_datetime(taxi["tpep_pickup_datetime"])
taxi = taxi[
    (taxi["tpep_pickup_datetime"].dt.year  == 2026) &
    (taxi["tpep_pickup_datetime"].dt.month == 1) &
    (taxi["fare_amount"]     > 0) &
    (taxi["trip_distance"]   > 0) &
    (taxi["passenger_count"] > 0)
].copy()

taxi["hour_bucket"] = taxi["tpep_pickup_datetime"].dt.floor("h")
hourly = (
    taxi.groupby("hour_bucket")
    .agg(
        trip_count     = ("VendorID",       "count"),
        avg_fare       = ("fare_amount",     "mean"),
        avg_distance   = ("trip_distance",   "mean"),
        avg_passengers = ("passenger_count", "mean")
    )
    .reset_index()
    .rename(columns={"hour_bucket": "datetime"})
)
print(f"✅ {len(taxi):,} trips → {len(hourly)} hourly records")

In [ ]:
# ── 2b. Temporal + lag features ────────────────────────────────────
hourly["hour"]        = hourly["datetime"].dt.hour
hourly["day_of_week"] = hourly["datetime"].dt.dayofweek
hourly["day"]         = hourly["datetime"].dt.day
hourly["is_weekend"]  = (hourly["day_of_week"] >= 5).astype(int)
hourly["is_rush_am"]  = ((hourly["hour"] >= 7)  & (hourly["hour"] <= 9)).astype(int)
hourly["is_rush_pm"]  = ((hourly["hour"] >= 17) & (hourly["hour"] <= 19)).astype(int)
hourly["is_night"]    = ((hourly["hour"] >= 22) | (hourly["hour"] <= 5)).astype(int)
# Cyclical encoding
hourly["hour_sin"]    = np.sin(2 * np.pi * hourly["hour"] / 24)
hourly["hour_cos"]    = np.cos(2 * np.pi * hourly["hour"] / 24)
hourly["dow_sin"]     = np.sin(2 * np.pi * hourly["day_of_week"] / 7)
hourly["dow_cos"]     = np.cos(2 * np.pi * hourly["day_of_week"] / 7)
# Lag features
hourly = hourly.sort_values("datetime").reset_index(drop=True)
hourly["lag_1h"]           = hourly["trip_count"].shift(1)
hourly["lag_24h"]          = hourly["trip_count"].shift(24)
hourly["lag_168h"]         = hourly["trip_count"].shift(168)
hourly["rolling_3h_mean"]  = hourly["trip_count"].rolling(3).mean().shift(1)
hourly["rolling_24h_mean"] = hourly["trip_count"].rolling(24).mean().shift(1)
print(f"✅ Temporal + lag features added. Shape: {hourly.shape}")

In [ ]:
# ── 2c. Weather ────────────────────────────────────────────────────
weather_cache = DATA_DIR / "weather_nyc_2026_01.csv"
if weather_cache.exists():
    weather = pd.read_csv(weather_cache, parse_dates=["datetime"])
    print(f"✅ Weather loaded from cache: {weather.shape}")
else:
    try:
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": 40.7128, "longitude": -74.0060,
            "start_date": "2026-01-01", "end_date": "2026-01-31",
            "hourly": "temperature_2m,precipitation,windspeed_10m,weathercode",
            "timezone": "America/New_York"
        }
        resp  = requests.get(url, params=params, timeout=30)
        wdata = resp.json()["hourly"]
        weather = pd.DataFrame({
            "datetime"        : pd.to_datetime(wdata["time"]),
            "temperature_c"   : wdata["temperature_2m"],
            "precipitation_mm": wdata["precipitation"],
            "windspeed_kmh"   : wdata["windspeed_10m"],
            "weather_code"    : wdata["weathercode"]
        })
        weather.to_csv(weather_cache, index=False)
        print(f"✅ Weather fetched: {weather.shape}")
    except Exception as e:
        print(f"⚠️  API failed — using mock data. ({e})")
        hours = pd.date_range("2026-01-01", "2026-01-31 23:00", freq="h")
        weather = pd.DataFrame({
            "datetime"        : hours,
            "temperature_c"   : np.random.normal(2, 5, len(hours)),
            "precipitation_mm": np.random.exponential(0.3, len(hours)),
            "windspeed_kmh"   : np.random.normal(15, 5, len(hours)).clip(0),
            "weather_code"    : np.random.choice([0,1,2,3,61,71], len(hours))
        })

weather["is_raining"]     = (weather["precipitation_mm"] > 0.5).astype(int)
weather["is_snowing"]     = weather["weather_code"].isin([71,73,75,77]).astype(int)
weather["is_bad_weather"] = ((weather["is_raining"]==1) | (weather["is_snowing"]==1)).astype(int)
display(weather.head(3))

In [ ]:
# ── 2d. Sentiment score ────────────────────────────────────────────
sentiment_map = {
    1: 1.0, 0: 0.0, -1: -1.0,
    "positive": 1.0, "neutral": 0.0, "negative": -1.0,
    "Positive": 1.0, "Neutral": 0.0, "Negative": -1.0
}
sentiment_col = "category"
scores = []
for path, name in [(REDDIT_PATH, "Reddit"), (TWITTER_PATH, "Twitter")]:
    df = pd.read_csv(path)
    if sentiment_col in df.columns:
        s = df[sentiment_col].map(sentiment_map).dropna().mean()
        scores.append(s)
        print(f"  {name} sentiment: {s:.4f}  ({len(df):,} records)")
global_sentiment = float(np.mean(scores)) if scores else 0.0
hourly["sentiment_score"] = global_sentiment
print(f"✅ Global sentiment score: {global_sentiment:.4f}")

In [ ]:
# ── 2e. Events flag ────────────────────────────────────────────────
events = pd.read_csv(EVENTS_PATH)
date_cols = [c for c in events.columns if any(
    kw in c.lower() for kw in ["date", "start", "begin"])]
if date_cols:
    events["event_date"] = pd.to_datetime(events[date_cols[0]], errors="coerce")
    event_days = set(
        events[(events["event_date"].dt.year==2026) &
               (events["event_date"].dt.month==1)]["event_date"].dt.date
    )
    hourly["has_event"] = hourly["datetime"].dt.date.apply(
        lambda d: 1 if d in event_days else 0)
    print(f"✅ Event days in Jan 2026: {len(event_days)}")
else:
    hourly["has_event"] = 0
    print("⚠️  No date column found — has_event set to 0")

In [ ]:
# ── 2f. Merge all & finalize ───────────────────────────────────────
weather["datetime"] = pd.to_datetime(weather["datetime"])
dataset = hourly.merge(weather, on="datetime", how="left").dropna().reset_index(drop=True)
dataset.to_csv(DATA_DIR / "merged_features_2026_01.csv", index=False)
print(f"✅ Final dataset: {dataset.shape}")
print(f"   Target stats → mean: {dataset['trip_count'].mean():.0f}  "
      f"std: {dataset['trip_count'].std():.0f}  "
      f"min: {dataset['trip_count'].min():.0f}  "
      f"max: {dataset['trip_count'].max():.0f}")

## 3. Train / Test Split (Chronological)

In [ ]:
ALL_FEATURES = [
    # Temporal
    "hour", "day_of_week", "day", "is_weekend",
    "is_rush_am", "is_rush_pm", "is_night",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    # Lag
    "lag_1h", "lag_24h", "lag_168h",
    "rolling_3h_mean", "rolling_24h_mean",
    # Weather
    "temperature_c", "precipitation_mm", "windspeed_kmh",
    "is_raining", "is_snowing", "is_bad_weather",
    # Events & sentiment
    "has_event", "sentiment_score",
    # Trip stats
    "avg_fare", "avg_distance", "avg_passengers",
]
TARGET = "trip_count"

FEATURES = [f for f in ALL_FEATURES if f in dataset.columns]
X = dataset[FEATURES]
y = dataset[TARGET]

split_idx   = int(len(dataset) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = dataset["datetime"].iloc[split_idx:]

print(f"Train: {len(X_train)} rows  ({dataset['datetime'].iloc[0].date()} → "
      f"{dataset['datetime'].iloc[split_idx-1].date()})")
print(f"Test : {len(X_test)}  rows  ({dataset['datetime'].iloc[split_idx].date()} → "
      f"{dataset['datetime'].iloc[-1].date()})")
print(f"Features used: {len(FEATURES)}")

## 4. Model Comparison
Compare four models on identical train/test splits to justify XGBoost selection.

In [ ]:
def evaluate(y_true, y_pred):
    """Return dict of evaluation metrics."""
    return {
        "RMSE" : round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        "MAE"  : round(mean_absolute_error(y_true, y_pred), 2),
        "R²"   : round(r2_score(y_true, y_pred), 4),
        "MAPE" : round(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100, 2),
    }

# Scale for Ridge only
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

candidate_models = {
    "Ridge (Baseline)": (Ridge(alpha=1.0), X_tr_sc, X_te_sc),
    "Random Forest"   : (RandomForestRegressor(n_estimators=200, max_depth=10,
                                               n_jobs=-1, random_state=42), X_train, X_test),
    "LightGBM"        : (lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                           num_leaves=63, random_state=42,
                                           verbosity=-1), X_train, X_test),
    "XGBoost"         : (xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                                          max_depth=6, subsample=0.8,
                                          colsample_bytree=0.8, random_state=42,
                                          verbosity=0), X_train, X_test),
}

comparison_results = {}
trained_models     = {}

for name, (mdl, Xtr, Xte) in candidate_models.items():
    print(f"Training {name}...", end=" ")
    mdl.fit(Xtr, y_train)
    pred = mdl.predict(Xte)
    comparison_results[name] = evaluate(y_test.values, pred)
    trained_models[name]     = (mdl, pred)
    print(f"RMSE={comparison_results[name]['RMSE']:.1f}  R²={comparison_results[name]['R²']:.4f}")

comparison_df = pd.DataFrame(comparison_results).T
print("\n" + "="*55)
print("           MODEL COMPARISON RESULTS")
print("="*55)
display(comparison_df.sort_values("RMSE"))

In [ ]:
# ── Visualize comparison ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics_to_plot = ["RMSE", "MAE", "R²"]
palette = ["#95a5a6", "#3498db", "#2ecc71", "#e74c3c"]

for ax, metric in zip(axes, metrics_to_plot):
    vals  = comparison_df[metric]
    bars  = ax.bar(vals.index, vals.values, color=palette, edgecolor="white", width=0.55)
    ax.set_title(f"{metric} by Model", fontsize=11)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01*bar.get_height(),
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Model Comparison — Test Set Performance", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: results/model_comparison.png")

## 5. TimeSeriesSplit Cross-Validation
5-fold time-series CV to get stable RMSE estimates with confidence intervals.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_models = {
    "Ridge"       : Ridge(alpha=1.0),
    "LightGBM"    : lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                       verbosity=-1, random_state=42),
    "XGBoost"     : xgb.XGBRegressor(n_estimators=300, learning_rate=0.05,
                                      max_depth=6, random_state=42, verbosity=0),
}

cv_results = {name: [] for name in cv_models}

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    sc = StandardScaler()
    X_tr_sc  = sc.fit_transform(X_tr)
    X_val_sc = sc.transform(X_val)

    for name, mdl in cv_models.items():
        Xtr_in = X_tr_sc if name == "Ridge" else X_tr
        Xval_in= X_val_sc if name == "Ridge" else X_val
        mdl.fit(Xtr_in, y_tr)
        pred = mdl.predict(Xval_in)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        cv_results[name].append(rmse)

print("\n" + "="*50)
print("    5-FOLD TIME-SERIES CROSS-VALIDATION RMSE")
print("="*50)
cv_summary = {}
for name, scores in cv_results.items():
    mean_rmse = np.mean(scores)
    std_rmse  = np.std(scores)
    cv_summary[name] = {"Mean RMSE": round(mean_rmse, 2),
                        "Std RMSE" : round(std_rmse, 2),
                        "Min RMSE" : round(min(scores), 2),
                        "Max RMSE" : round(max(scores), 2)}
    print(f"  {name:<15}: {mean_rmse:.2f} ± {std_rmse:.2f}")

display(pd.DataFrame(cv_summary).T)

In [ ]:
# ── CV results visualization ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(1, 6)
colors = {"Ridge": "#95a5a6", "LightGBM": "#2ecc71", "XGBoost": "#e74c3c"}

for name, scores in cv_results.items():
    ax.plot(x, scores, marker="o", label=name,
            color=colors[name], linewidth=2, markersize=7)

ax.set_title("5-Fold Time-Series CV RMSE per Fold", fontsize=12)
ax.set_xlabel("Fold")
ax.set_ylabel("RMSE (trips/hour)")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cv_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: results/cv_results.png")

## 6. Hyperparameter Tuning (Optuna)
Bayesian optimization of XGBoost hyperparameters using 3-fold TimeSeriesSplit.

In [ ]:
if OPTUNA_AVAILABLE:
    def objective(trial):
        params = {
            "n_estimators"     : trial.suggest_int("n_estimators", 100, 700),
            "max_depth"        : trial.suggest_int("max_depth", 3, 9),
            "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight" : trial.suggest_int("min_child_weight", 1, 10),
            "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
            "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
            "random_state"     : 42,
            "verbosity"        : 0,
        }
        cv = TimeSeriesSplit(n_splits=3)
        scores = []
        for tr_idx, val_idx in cv.split(X_train):
            m = xgb.XGBRegressor(**params)
            m.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            pred = m.predict(X_train.iloc[val_idx])
            scores.append(np.sqrt(mean_squared_error(y_train.iloc[val_idx], pred)))
        return np.mean(scores)

    study = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    print(f"\n✅ Optuna finished — best CV RMSE: {study.best_value:.2f}")
    print("Best hyperparameters:")
    for k, v in study.best_params.items():
        print(f"  {k:<25}: {v}")

    BEST_PARAMS = {**study.best_params, "random_state": 42, "verbosity": 0}
else:
    print("⚠️  Optuna not available — using default params")
    BEST_PARAMS = {
        "n_estimators": 500, "learning_rate": 0.05, "max_depth": 6,
        "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5,
        "reg_alpha": 0.1, "reg_lambda": 1.0,
        "random_state": 42, "verbosity": 0
    }

## 7. Final Model Training & Evaluation

In [ ]:
# ── Train final XGBoost with best params ──────────────────────────
final_model = xgb.XGBRegressor(
    **BEST_PARAMS,
    eval_metric="rmse",
    early_stopping_rounds=30,
)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)
y_pred = final_model.predict(X_test)
metrics = evaluate(y_test.values, y_pred)

print("\n" + "="*45)
print("    FINAL MODEL EVALUATION")
print("="*45)
for k, v in metrics.items():
    print(f"  {k:<8}: {v}")
print("="*45)

pd.DataFrame([metrics]).to_csv(RESULTS_DIR / "metrics.csv", index=False)
print("✅ Metrics saved")

In [ ]:
# ── Plot: Predicted vs Actual ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(dates_test.values, y_test.values,
             label="Actual",    color="steelblue", lw=1.2, alpha=0.9)
axes[0].plot(dates_test.values, y_pred,
             label="Predicted", color="coral",     lw=1.2, alpha=0.9, ls="--")
axes[0].set_title("Actual vs Predicted Hourly Taxi Demand (Test Set)", fontsize=13)
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Trip Count")
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
axes[0].tick_params(axis='x', rotation=45)

axes[1].scatter(y_test, y_pred, alpha=0.3, color="steelblue", s=15)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[1].plot(lims, lims, 'r--', lw=1.5, label="Perfect prediction")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].set_title(f"Scatter — R² = {metrics['R²']:.4f}", fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "predicted_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()

# Training loss curve
evals = final_model.evals_result()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(evals["validation_0"]["rmse"], label="Train RMSE", color="steelblue")
ax.plot(evals["validation_1"]["rmse"], label="Test RMSE",  color="coral")
ax.axvline(final_model.best_iteration, color="grey", ls=":",
           label=f"Best iter ({final_model.best_iteration})")
ax.set_title("XGBoost Training Loss Curve", fontsize=12)
ax.set_xlabel("Boosting Round")
ax.set_ylabel("RMSE")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plots saved")

## 8. Ablation Study
Incrementally add each data modality to measure its individual contribution.

In [ ]:
TEMPORAL_FEATS   = ["hour","day_of_week","day","is_weekend",
                    "is_rush_am","is_rush_pm","is_night",
                    "hour_sin","hour_cos","dow_sin","dow_cos"]
LAG_FEATS        = ["lag_1h","lag_24h","lag_168h",
                    "rolling_3h_mean","rolling_24h_mean"]
WEATHER_FEATS    = ["temperature_c","precipitation_mm","windspeed_kmh",
                    "is_raining","is_snowing","is_bad_weather"]
EVENT_FEATS      = ["has_event"]
SENTIMENT_FEATS  = ["sentiment_score"]

ablation_configs = {
    "① Temporal only"         : TEMPORAL_FEATS,
    "② + Lag features"        : TEMPORAL_FEATS + LAG_FEATS,
    "③ + Weather"             : TEMPORAL_FEATS + LAG_FEATS + WEATHER_FEATS,
    "④ + Events"              : TEMPORAL_FEATS + LAG_FEATS + WEATHER_FEATS + EVENT_FEATS,
    "⑤ Full (+ Sentiment)"    : TEMPORAL_FEATS + LAG_FEATS + WEATHER_FEATS + EVENT_FEATS + SENTIMENT_FEATS,
}

ablation_results = {}
print("Running ablation study...")
print("-" * 60)
for exp_name, feat_list in ablation_configs.items():
    avail = [f for f in feat_list if f in X_train.columns]
    m = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05,
                          max_depth=6, random_state=42, verbosity=0)
    m.fit(X_train[avail], y_train)
    pred = m.predict(X_test[avail])
    res  = evaluate(y_test.values, pred)
    ablation_results[exp_name] = res
    print(f"  {exp_name:<30}  RMSE={res['RMSE']:>8.2f}  R²={res['R²']:.4f}  "
          f"({len(avail)} features)")

ablation_df = pd.DataFrame(ablation_results).T
display(ablation_df)

In [ ]:
# ── Ablation visualization ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
labels = [k.split(" ",1)[1] for k in ablation_df.index]   # remove emoji prefix

# RMSE — lower is better
axes[0].barh(labels, ablation_df["RMSE"].values,
             color="#5b9bd5", edgecolor="white")
for i, v in enumerate(ablation_df["RMSE"].values):
    axes[0].text(v + 1, i, f"{v:.1f}", va="center", fontsize=9)
axes[0].set_title("Ablation Study — RMSE (↓ lower is better)", fontsize=11)
axes[0].set_xlabel("RMSE (trips/hour)")
axes[0].invert_yaxis()

# R² — higher is better
axes[1].barh(labels, ablation_df["R²"].values,
             color="#70ad47", edgecolor="white")
for i, v in enumerate(ablation_df["R²"].values):
    axes[1].text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)
axes[1].set_title("Ablation Study — R² (↑ higher is better)", fontsize=11)
axes[1].set_xlabel("R²")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "ablation_study.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: results/ablation_study.png")

# Print improvement summary
rmse_vals = ablation_df["RMSE"].values
print("\nIncremental RMSE improvement from each modality:")
keys = list(ablation_configs.keys())
for i in range(1, len(keys)):
    delta = rmse_vals[i-1] - rmse_vals[i]
    pct   = delta / rmse_vals[i-1] * 100
    modality = keys[i].split("+")[-1].strip()
    print(f"  Adding {modality:<20}: RMSE ↓ {delta:.2f}  ({pct:.1f}% improvement)")

## 9. Error Analysis
Understand when and where the model fails.

In [ ]:
error_df = pd.DataFrame({
    "datetime"   : dates_test.values,
    "actual"     : y_test.values,
    "predicted"  : y_pred,
    "error"      : y_test.values - y_pred,
    "abs_error"  : np.abs(y_test.values - y_pred),
    "pct_error"  : np.abs(y_test.values - y_pred) / (y_test.values + 1e-8) * 100,
})
error_df["hour"]       = pd.to_datetime(error_df["datetime"]).dt.hour
error_df["day_of_week"]= pd.to_datetime(error_df["datetime"]).dt.dayofweek
error_df["is_weekend"] = (error_df["day_of_week"] >= 5).astype(int)

print("=" * 50)
print("  ERROR ANALYSIS SUMMARY")
print("=" * 50)
print(f"  Mean abs error   : {error_df['abs_error'].mean():.1f} trips")
print(f"  Median abs error : {error_df['abs_error'].median():.1f} trips")
print(f"  Max abs error    : {error_df['abs_error'].max():.1f} trips")
print(f"  % errors > 500   : {(error_df['abs_error'] > 500).mean()*100:.1f}%")
print()

# Worst 5 predictions
print("  Top 5 worst predictions:")
display(error_df.nlargest(5, 'abs_error')[["datetime","actual","predicted","abs_error","pct_error"]])

In [ ]:
# ── Error breakdown plots ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# By hour of day
hourly_err = error_df.groupby("hour")["abs_error"].mean()
axes[0,0].bar(hourly_err.index, hourly_err.values, color="steelblue", edgecolor="white")
axes[0,0].set_title("Mean Absolute Error by Hour of Day", fontsize=11)
axes[0,0].set_xlabel("Hour")
axes[0,0].set_ylabel("MAE (trips)")
axes[0,0].set_xticks(range(0, 24, 2))

# By day of week
dow_err    = error_df.groupby("day_of_week")["abs_error"].mean()
dow_labels = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
axes[0,1].bar(dow_labels, dow_err.values,
              color=["#e74c3c" if i>=5 else "steelblue" for i in range(7)],
              edgecolor="white")
axes[0,1].set_title("Mean Absolute Error by Day of Week", fontsize=11)
axes[0,1].set_ylabel("MAE (trips)")

# Weekday vs Weekend
wk_err = error_df.groupby("is_weekend")["abs_error"].agg(["mean","std"])
axes[1,0].bar(["Weekday","Weekend"], wk_err["mean"].values,
              yerr=wk_err["std"].values, color=["steelblue","coral"],
              edgecolor="white", capsize=5)
axes[1,0].set_title("MAE: Weekday vs Weekend", fontsize=11)
axes[1,0].set_ylabel("MAE (trips)")

# Residual distribution
axes[1,1].hist(error_df["error"].values, bins=40,
               color="steelblue", edgecolor="white", alpha=0.8)
axes[1,1].axvline(0, color="red", ls="--", lw=1.5)
axes[1,1].axvline(error_df["error"].mean(), color="orange", ls="--",
                  lw=1.5, label=f"Mean={error_df['error'].mean():.1f}")
axes[1,1].set_title("Residual Distribution", fontsize=11)
axes[1,1].set_xlabel("Residual (Actual − Predicted)")
axes[1,1].legend()

plt.suptitle("Error Analysis — Where Does the Model Fail?", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: results/error_analysis.png")

## 10. SHAP Interpretability

In [ ]:
if SHAP_AVAILABLE:
    explainer = shap.TreeExplainer(final_model)
    shap_vals = explainer.shap_values(X_test)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar summary
    plt.sca(axes[0])
    shap.summary_plot(shap_vals, X_test, plot_type="bar",
                      show=False, max_display=15)
    axes[0].set_title("SHAP Feature Importance (Mean |SHAP|)", fontsize=11)

    # Beeswarm
    plt.sca(axes[1])
    shap.summary_plot(shap_vals, X_test, show=False, max_display=12)
    axes[1].set_title("SHAP Beeswarm — Feature Impact Direction", fontsize=11)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved: results/shap_summary.png")
else:
    print("⚠️  SHAP not available. Install with: pip install shap")

## 11. Save Final Model & Summary

In [ ]:
joblib.dump(final_model, MODELS_DIR / "xgb_demand_model.pkl")
joblib.dump(FEATURES,    MODELS_DIR / "feature_cols.pkl")

# Save all results tables
comparison_df.to_csv(RESULTS_DIR / "model_comparison.csv")
ablation_df.to_csv(RESULTS_DIR   / "ablation_study.csv")
pd.DataFrame(cv_summary).T.to_csv(RESULTS_DIR / "cv_results.csv")
error_df.to_csv(RESULTS_DIR      / "error_analysis.csv", index=False)

print("=" * 55)
print("  TRAINING COMPLETE — ALL OUTPUTS SAVED")
print("=" * 55)
print(f"\n  Final Model Metrics:")
for k, v in metrics.items():
    print(f"    {k:<8}: {v}")
print(f"\n  Saved files:")
print(f"    models/xgb_demand_model.pkl")
print(f"    results/model_comparison.png / .csv")
print(f"    results/ablation_study.png / .csv")
print(f"    results/cv_results.png / .csv")
print(f"    results/error_analysis.png / .csv")
print(f"    results/predicted_vs_actual.png")
print(f"    results/training_loss_curve.png")
print(f"    results/shap_summary.png")
print("=" * 55)